LLM Zoomcamp 1.12 — Quick RAG Revision (Optional)


In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Load environment variables from .env file
openai_client = OpenAI()  # Initialize the OpenAI client

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [4]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

sqlite_index.search("how do I join the course")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/

In [21]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    instructions=instructions,
)

In [22]:
answer = assistant.rag('How do I run Ollama locally')
print(answer)

To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download the `.pkg` and install it
     - **Windows**: download the `.msi` and install it
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. **Test the local server**
   ```bash
   curl http://localhost:11434
   ```
   You should see a response like:
   ```json
   {"models": [...]}  
   ```

4. **Use it from Python**
   Install the client:
   ```bash
   pip install ollama
   ```
   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```

If you get a connection refused error while prompting Ollama in the hom

LLM Zoomcamp 1.13 — Function Calling

In [37]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—if the course is still open, you can usually join it.\n\nWhat you should do next:\n1. Check the course page for a **“Join,” “Enroll,” “Register,”** or **“Request access”** button.\n2. See whether there are any **prerequisites**, **registration deadlines**, or **capacity limits**.\n3. If you don’t see a way to enroll, contact the **course instructor, coordinator, or support team**.\n\nIf you want, I can help you draft a short message to ask for enrollment.'

In [33]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [38]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [39]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join anytime enrollment late registration"}', call_id='call_QqwGUMlKFYSxswpeP8nbVmbu', name='search', type='function_call', id='fc_0d1830bfeff372af006af598b31b1887d0be9c1b3120f987ad', async_=None, caller=None, namespace=None, status='completed')]

In [40]:
import json

call = response.output[0]
args = json.loads(call.arguments)

In [48]:
print(call)

ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join anytime enrollment late registration"}', call_id='call_QqwGUMlKFYSxswpeP8nbVmbu', name='search', type='function_call', id='fc_0d1830bfeff372af006af598b31b1887d0be9c1b3120f987ad', async_=None, caller=None, namespace=None, status='completed')


In [49]:
call.name

'search'

In [41]:
print(args)

{'query': 'Can I join the course late discovered the course join anytime enrollment late registration'}


In [42]:
results = search(**args)
result_json = json.dumps(results, indent=2)

In [43]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",
    "answer": "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."
  },
  {
    "id": "cdc3b285e5",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Can I submit homework after the deadline, or get a d

In [51]:
function_call_output = {
    "type": "function_call_Output",
    "call_id":call.call_id,
    "output": result_json
}

In [52]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    function_call_outputs=[function_call_output],
    tools=[search_tool]
)

TypeError: Responses.create() got an unexpected keyword argument 'function_call_outputs'

In [44]:
messages.extend(response.output)

In [45]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [46]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.'

In [47]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(687, 32)

In [20]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


1. make a call to LLM <-- 1st LLM Call
2. LLM decided to invoke search('params')
3. we invoke the search, we have the results
4. LLM processes the search results <-- 2nd LLM Call
5. LLM gives the answer
